In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from rapidfuzz import process
import pickle
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle

In [ ]:
# Count number of species per aviary 
metadata_path = 'metadata_aviaries/'
species_df = pd.read_csv(metadata_path + 'Avieries_obsolete.csv', encoding='latin-1')

number_species = species_df.groupby('Aviary')['Common name'].nunique()
number_individuals = species_df.groupby('Aviary')['Total count'].sum()
species_list = species_df.groupby('Aviary')['Common name'].apply(list)
individuals_list = species_df.groupby('Aviary')['Total count'].apply(list)
individuals_gender_list = species_df.groupby('Aviary')['Genders (male.female.unknown)'].apply(list)
metadata_paths = species_df.groupby('Aviary')['preprocessed data'].first()  

combined_df = pd.DataFrame({'species': species_list, 'individuals': individuals_list, 'individuals_genders (m.f.u)': individuals_gender_list, 'Number of Species': number_species, 'Number of Individuals': number_individuals, "preprocessed data": metadata_paths})
combined_df = combined_df.dropna() # Some aviaries don't have metadata files, so we drop those rows for now

# Link preprocessed data path to metadata path as they are not in the same format
aviaries_metadata = [f for f in os.listdir(metadata_path) if f.endswith('.xlsx')]
for idx, row in combined_df.iterrows():
    avirary_path = row["preprocessed data"]
    best_match = process.extractOne(avirary_path, aviaries_metadata)
    if best_match and best_match[1] > 80:
        file_path = best_match[0]

    combined_df.loc[row.name, "Metadata_filename"] = file_path

combined_df.reset_index(inplace=True)
display(combined_df)
# combined_df.to_csv("general_aviary_data.csv", index=False)


,Aviary,species,individuals,individuals_genders (m.f.u),Number of Species,Number of Individuals,preprocessed data,Metadata_filename
0,"Avifauna, Cuba aviary","[American Flamingo, Picazuro pigeon, Abdim's s...","[160, 2, 1, 20, 20, 18, 4, 22]","[68.62.30, 1.1.0, 0.1.0, 6.9.5, 2.4.14, 6.6.6,...",8,247,fl_avifauna_flamingos_sept2025_data,fl_avifauna_flamingos_sept25_data_meta.xlsx
1,"Avifauna, Umvikeli aviary","[Eastern Crested Guineafowl, Yellow-necked fra...","[6, 3, 3, 2, 23, 2, 2, 1, 2, 3, 1, 2, 21]","[2.3.1, 1.2.0, 1.2.0, 1.1.0, 11.11.1, 1.1.0, 1...",13,71,fl_avifauna_vultures_sept2025_data,fl_avifauna_vultures_sept25_data_meta.xlsx
2,"Avifauna, storage","[Black-bellied whistling duck, Inca tern, Scar...","[7, 8, 19, 2]","[4.3.0, 0.0.8, 1.5.13, 1.1.0]",4,36,fl_avifauna_storage_sept2025_data,fl_avifauna_vultures_sept25_data_meta.xlsx
3,"Beekse Bergen, Aviary 2",[Griffon vulture],[2],[1.1],1,2,fl_beekse_bergen_20250404,fl_beekse_bergen_20250404_meta.xlsx
4,"Beekse Bergen, Aviary 3","[Sacred ibis, Rüppell's vulture, Secretarybird...","[2, 5, 2, 3, 2, 4, 141, 4]","[0.1, 2.3, 1.1, 2.1, 1.1.0, 1.2.1, 38.54.49, 2...",8,163,fl_beekse_bergen_20250404,fl_beekse_bergen_20250404_meta.xlsx
5,"Beekse Bergen, Aviary 4 (Africa)","[African spoonbill, Spotted thick-knee, Eurasi...","[9, 5, 3, 3, 2, 1, 153, 6]","[4.5.0, 2.3.0, 2.1.0, 1.1.1, 1.1.0, 0.1.0, 61....",8,182,fl_beekse_bergen_20250412,fl_beekse_bergen_20250412_meta.xlsx
6,"Beekse Bergen, aviary 5","[Greater flamingo, Hadada Ibis]","[107, 6]","[56.51.0, 1.4.1]",2,113,fl_beekse_bergen_flaminogos_sept2025_data,fl_beekse_bergen_20250412_meta.xlsx
7,"Blijdorp, Flamingos","[Greater flamingo, Baer's pochard, Northern sh...","[195, 3, 2, 4]","[0.0.195, 0.0.3, 0.0.2, 0.0.4]",4,204,fl_blijdorp_flamingos_dec2025_data,fl_blijdorp_flamingos_dec2025_data_meta.xlsx
8,"Cologne, Flamingos",[American Flamingo],[153],[77.76.0],1,153,fl_cologne_zoo_flamingos_nov2025_data,fl_cologne_zoo_flamingos_nov2025_data_meta.xlsx
9,"Cologne, Hippodom","[Taveta Golden Weaver, White-browed Coucal, Wh...","[22, 1, 3, 33, 5, 2, 2, 5, 4, 2, 2, 5, 2, 3, 3...","[11.12.0, 1.0.0, 2.1.0, 15.18.0, 2.3.0, 1.0.0,...",17,98,fl_cologne_zoo_indoors_long_nov2025_data,fl_cologne_zoo_flamingos_nov2025_data_meta.xlsx


In [3]:
# count number of bird call per aviary meta data file
# We don't have all the metadata files, so we can't get the info for all aviaries in the cell above

aviaries_metadata = [f for f in os.listdir(metadata_path) if f.endswith('.xlsx')]
#EDA_df = pd.DataFrame(columns=["file_name", "call_count", "total_entries", "Ratio", "Number_Species", "Number_Individuals", "start_date", "end_date", "Duration_days", "Unique_days"])
EDA_df = pd.DataFrame(columns=["Zoo name", "call_count", "total_entries", "Number_Species", "Number_Individuals", "Duration_days"])

for file in aviaries_metadata:
    print(f"Processing file: {file}")

    try:
        df = pd.read_excel(metadata_path + file)

        call_count = df["fusion_model_prediction"].notnull().sum()
        
        number_species = combined_df[combined_df["Metadata_filename"] == file]["Number of Species"].iloc[0] if file in combined_df["Metadata_filename"].values else 0
        number_individuals = combined_df[combined_df["Metadata_filename"] == file]["Number of Individuals"].iloc[0] if file in combined_df["Metadata_filename"].values else 0

        start_date = df["datetime"].min()
        end_date = df["datetime"].max()
        unique_days = df["datetime"].dt.date.nunique()

        zoo_name = combined_df[combined_df["Metadata_filename"] == file]["Aviary"].iloc[0] if file in combined_df["Metadata_filename"].values else "Unknown"

        new_row = {
            "Zoo name": zoo_name,
            "call_count": call_count,
            "total_entries": len(df),
            #"Ratio": call_count / len(df),
            "Number_Species": number_species,
            "Number_Individuals": number_individuals,
            #"start_date": start_date,
            #"end_date": end_date,
            "Duration_days": (end_date - start_date).days
            #"Unique_days": unique_days
        }
        
        EDA_df.loc[len(EDA_df)] = new_row

    except Exception as e:
        print(f"Error processing file {file}: {e}")
            
EDA_df

Processing file: fl_avifauna_flamingos_sept25_data_meta.xlsx
Processing file: fl_avifauna_storage_ibises_sept25_data_meta.xlsx
Processing file: fl_avifauna_vultures_sept25_data_meta.xlsx
Processing file: fl_beekse_bergen_20250404_meta.xlsx
Processing file: fl_beekse_bergen_20250412_meta.xlsx
Processing file: fl_blijdorp_flamingos_dec2025_data_meta.xlsx
Processing file: fl_cologne_zoo_flamingos_nov2025_data_meta.xlsx


KeyboardInterrupt: 

In [ ]:
EDA_df.drop(EDA_df[EDA_df["Zoo name"] == "Unknown"].index, inplace=True)
EDA_df.to_csv("Aviary_EDA.csv", index=False)

In [ ]:
# PLot aviary diversity for each zoo:
plt.figure(figsize=(12, 6))
plt.bar(EDA_df["Zoo name"], EDA_df["Number_Species"], color='skyblue')
plt.xlabel("Zoo Name")  
plt.ylabel("Number of Species")
plt.title("Aviary Diversity by Zoo")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined